# 002 - Build Main Anime Dataset From Raw Sources

This notebook turns the raw caches from `001` into `data/processed/anime_dataset.csv/json`.

Important design changes:

- MAL/Jikan remains the authority for public catalog metrics: score, scored_by, rank, popularity, members, favorites, image, title, and URL.
- AniList is the authority for `genres`, `tags`, `tag_weights`, `explicit_tags`, and `explicit_tag_weights`.
- MAL `Erotica`/`Hentai` and AniDB `Loli` remain special caveats.
- AniDB is a final fallback, not the main tag system.
- Disagreements are preserved as comparison columns such as `episodes_mal`, `episodes_anilist`, `episodes_anidb`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

SCRIPT = BASE_DIR / "src" / "02_build_anime_dataset.py"
DATASET = BASE_DIR / "data" / "processed" / "anime_dataset.csv"
SUMMARY = BASE_DIR / "data" / "build" / "dataset_build_summary.json"
DISCREPANCIES = BASE_DIR / "data" / "build" / "dataset_source_discrepancies.csv"



def run_streaming(command, cwd=None):
    """Run a script and print stdout/stderr line-by-line while it is still running."""
    cwd = cwd or globals().get("BASE_DIR") or globals().get("ROOT") or Path.cwd()
    command = [str(part) for part in command]
    print("Running:", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code


## Build Dataset

This is deterministic from the raw caches. If you do not fetch new raw source data, rerunning this cell should produce the same catalog.

In [ ]:
RUN_BUILD = True
if RUN_BUILD:
    run_streaming([sys.executable, str(SCRIPT)])


## Build Summary and Source Disagreements

Disagreements are not errors by themselves. They are review targets. Examples: MAL may split movies/specials differently from AniList or AniDB, and airing episode counts may lag across sources.

In [ ]:
df = pd.read_csv(DATASET)
discrepancies = pd.read_csv(DISCREPANCIES) if DISCREPANCIES.exists() else pd.DataFrame()

if SUMMARY.exists():
    display(json.loads(SUMMARY.read_text(encoding="utf-8")))

display(df.head())
display(df[['mal_id', 'anilist_id', 'anidb_id']].notna().mean().rename('coverage').to_frame())
display(discrepancies.head(50))

## Quick Label Checks

Genres and tags should now be AniList-first. Rows with no AniList match intentionally have blank `genres` and `tags` rather than falling back to noisy AniDB tags.

In [ ]:
label_cols = ['genres', 'tags', 'explicit_tags', 'demographics']
missing_labels = []
for col in label_cols:
    mask = df[col].isna() | df[col].astype(str).str.strip().isin(['', 'nan', 'None'])
    missing_labels.append({'column': col, 'missing_rows': int(mask.sum()), 'missing_pct': round(mask.mean(), 4)})
display(pd.DataFrame(missing_labels))

display(df.loc[df['anilist_id'].isna(), ['mal_id', 'title', 'type', 'status', 'score']].head(25))